# Marketing Budget Simulation Analysis

This notebook demonstrates how to use the simulation engine programmatically and analyze results.

In [ ]:
import sys
sys.path.append('..')

from simulation_engine import MarketingSimulation, create_budget
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

## Running a Full 8-Quarter Simulation

In [ ]:
# Initialize simulation
sim = MarketingSimulation()

# 2021 Strategy (Q1-Q4): Consistent allocation
budget_2021 = create_budget(
    discover_var=46, discover_fix=6,
    try_var=34, try_fix=6,
    buy_var=94, buy_fix=6,
    use_var=36, use_fix=6,
    renew_var=12, renew_fix=4
)

# Run 2021
results_2021 = []
for q in range(4):
    result = sim.run_quarter(budget_2021)
    results_2021.append(result)
    print(f"{result['quarter']}: ${result['net_arr']}M (ROI: {result['roi']:.2f})")

In [ ]:
# 2022 Strategy Q1-Q3: Heavy fixed investment
budget_2022_q1_q3 = create_budget(
    discover_var=56, discover_fix=12,
    try_var=46, try_fix=12,
    buy_var=78, buy_fix=18,
    use_var=38, use_fix=12,
    renew_var=16, renew_fix=12
)

# Run Q1-Q3 2022
results_2022 = []
for q in range(3):
    result = sim.run_quarter(budget_2022_q1_q3)
    results_2022.append(result)
    print(f"{result['quarter']}: ${result['net_arr']}M (ROI: {result['roi']:.2f})")

In [ ]:
# 2022 Q4: Optimize with more variable spend
budget_2022_q4 = create_budget(
    discover_var=58, discover_fix=10,
    try_var=48, try_fix=10,
    buy_var=82, buy_fix=12,
    use_var=41, use_fix=10,
    renew_var=19, renew_fix=10
)

result = sim.run_quarter(budget_2022_q4)
results_2022.append(result)
print(f"{result['quarter']}: ${result['net_arr']}M (ROI: {result['roi']:.2f})")

## Analyze Results

In [ ]:
# Combine all results
all_results = results_2021 + results_2022
df = pd.DataFrame(all_results)

# Calculate totals
total_2021 = df[df['quarter'].str.contains('21')]['net_arr'].sum()
total_2022 = df[df['quarter'].str.contains('22')]['net_arr'].sum()

print(f"\n2021 Total: ${total_2021:.0f}M (Target: $2000M = {total_2021/2000*100:.1f}%)")
print(f"2022 Total: ${total_2022:.0f}M (Target: $2500M = {total_2022/2500*100:.1f}%)")
print(f"Combined Total: ${total_2021 + total_2022:.0f}M (Target: $4500M = {(total_2021 + total_2022)/4500*100:.1f}%)")

df

## Visualize Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Net ARR Trend
axes[0, 0].plot(df['quarter'], df['net_arr'], marker='o', linewidth=2, markersize=8)
axes[0, 0].axhline(y=500, color='r', linestyle='--', label='2021 Target')
axes[0, 0].axhline(y=625, color='orange', linestyle='--', label='2022 Target')
axes[0, 0].set_title('Net-New ARR by Quarter', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('ARR ($M)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# ROI Trend
axes[0, 1].plot(df['quarter'], df['roi'], marker='s', color='green', linewidth=2, markersize=8)
axes[0, 1].set_title('ROI by Quarter', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('ROI')
axes[0, 1].grid(True, alpha=0.3)

# Funnel Metrics
axes[1, 0].plot(df['quarter'], df['visitors'], marker='o', label='Visitors (M)')
axes[1, 0].plot(df['quarter'], df['trialists'], marker='s', label='Trialists (M)')
axes[1, 0].plot(df['quarter'], df['orders'], marker='^', label='Orders (M)')
axes[1, 0].set_title('Funnel Metrics', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Count (Millions)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Retention Rate
axes[1, 1].plot(df['quarter'], df['retention_rate'], marker='D', color='purple', linewidth=2, markersize=8)
axes[1, 1].axhline(y=70, color='r', linestyle='--', label='Exit Criteria (70%)')
axes[1, 1].set_title('Retention Rate by Quarter', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Retention %')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Compare Gross ARR vs Cancel ARR

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(df))
width = 0.35

ax.bar([i - width/2 for i in x], df['gross_arr'], width, label='Gross ARR', color='green', alpha=0.7)
ax.bar([i + width/2 for i in x], df['cancel_arr'], width, label='Cancel ARR', color='red', alpha=0.7)

ax.set_xlabel('Quarter')
ax.set_ylabel('ARR ($M)')
ax.set_title('Gross ARR vs Cancel ARR', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df['quarter'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nChurn Improvement:")
print(f"Q1 2021 Cancel ARR: ${df.iloc[0]['cancel_arr']:.0f}M")
print(f"Q4 2022 Cancel ARR: ${df.iloc[-1]['cancel_arr']:.0f}M")
print(f"Improvement: ${df.iloc[-1]['cancel_arr'] - df.iloc[0]['cancel_arr']:.0f}M ({(df.iloc[-1]['cancel_arr']/df.iloc[0]['cancel_arr']-1)*100:.1f}%)")

## Test Different Strategies

In [ ]:
# Compare different Q1 2021 strategies
strategies = {
    'Balanced': create_budget(46, 6, 34, 6, 94, 6, 36, 6, 12, 4),
    'Heavy BUY': create_budget(40, 5, 30, 5, 120, 5, 30, 5, 15, 5),
    'High Fixed': create_budget(40, 15, 30, 15, 80, 20, 30, 15, 10, 10),
    'Retention Focus': create_budget(45, 5, 35, 5, 85, 5, 35, 5, 25, 10)
}

results_comparison = {}

for name, budget in strategies.items():
    sim_test = MarketingSimulation()
    result = sim_test.run_quarter(budget)
    results_comparison[name] = result['net_arr']

# Plot comparison
plt.figure(figsize=(10, 6))
plt.bar(results_comparison.keys(), results_comparison.values(), color=['blue', 'green', 'orange', 'purple'], alpha=0.7)
plt.axhline(y=500, color='r', linestyle='--', label='Target')
plt.ylabel('Net-New ARR ($M)')
plt.title('Q1 2021 Strategy Comparison', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')

for i, (name, value) in enumerate(results_comparison.items()):
    plt.text(i, value + 10, f'${value:.0f}M', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nStrategy Rankings:")
for i, (name, arr) in enumerate(sorted(results_comparison.items(), key=lambda x: x[1], reverse=True), 1):
    print(f"{i}. {name}: ${arr:.0f}M ({arr/500*100:.1f}% of target)")

## Export Results to Excel

In [ ]:
# Export to Excel
with pd.ExcelWriter('simulation_results.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Quarterly Results', index=False)
    
    # Summary sheet
    summary = pd.DataFrame({
        'Metric': ['2021 Total ARR', '2022 Total ARR', 'Combined Total', 
                   '2021 Achievement %', '2022 Achievement %', 'Overall Achievement %'],
        'Value': [
            f'${total_2021:.0f}M',
            f'${total_2022:.0f}M',
            f'${total_2021 + total_2022:.0f}M',
            f'{total_2021/2000*100:.1f}%',
            f'{total_2022/2500*100:.1f}%',
            f'{(total_2021 + total_2022)/4500*100:.1f}%'
        ]
    })
    summary.to_excel(writer, sheet_name='Summary', index=False)

print("Results exported to simulation_results.xlsx")